# GraphRAG Generation POC-3: Claude API Integration

## Suggesting Nodes and Edges Iteratively with topologic_fast

This notebook demonstrates how to use Large Language Models (specifically Claude) to iteratively build a topological graph representing a residential floor plan.

### Concept

**GraphRAG** (Graph Retrieval-Augmented Generation) combines:
1. **Knowledge Base**: Embeddings of typical room adjacency patterns
2. **Retrieval**: Finding relevant patterns for the current graph state
3. **LLM Generation**: Using Claude to suggest graph expansions
4. **Topology**: Building the graph using topologic_fast

### Flow

1. User specifies requirements (e.g., "three bedrooms and two bathrooms")
2. System retrieves relevant adjacency patterns from knowledge base
3. Claude suggests nodes (rooms) and edges (connections) to add
4. topologic_fast builds and manages the graph
5. Visualization shows the evolving floor plan

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import os
import re
import math
import random
import numpy as np

print(f"topologic_fast version: {tf.__version__}")

## Step 1: Setup Embedding Model

We use sentence-transformers for creating embeddings of room adjacency patterns.

In [ ]:
# Try to load sentence transformers
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    
    MODEL_NAME = "all-MiniLM-L6-v2"
    print(f"Loading SentenceTransformer model: {MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME)
    print("SentenceTransformer model loaded successfully")
    USE_EMBEDDINGS = True
except ImportError:
    print("sentence-transformers not available. Using fallback similarity.")
    USE_EMBEDDINGS = False
    model = None

## Step 2: Configure LLM Backend

This notebook uses Claude (Anthropic) as the LLM backend. You'll need an API key.

In [ ]:
# Check for API key
api_key = os.environ.get("ANTHROPIC_API_KEY")

if not api_key:
    print("ANTHROPIC_API_KEY not found in environment.")
    print("Please enter your Anthropic API key:")
    api_key = input("API Key: ").strip()
    
if api_key:
    print("API key configured.")
    os.environ["ANTHROPIC_API_KEY"] = api_key
else:
    print("Warning: No API key provided. LLM calls will fail.")

In [ ]:
# Setup Claude client
LLM_MODEL = os.getenv("LLM_MODEL", "claude-3-5-sonnet-20241022")

try:
    import anthropic
    client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", api_key))
    print(f"Using Claude model: {LLM_MODEL}")
    USE_LLM = True
except ImportError:
    print("anthropic library not installed. Install with: pip install anthropic")
    print("Using mock LLM responses instead.")
    client = None
    USE_LLM = False

## Step 3: Knowledge Base

Define typical room adjacency patterns and area hints.

In [ ]:
# Known graph patterns (name, list of (room_type, [adjacent_types]))
known_graphs = [
    ("basic_flat",
     [("living room", ["dining room", "kitchen", "hallway"]),
      ("kitchen", ["dining room", "living room"]),
      ("bedroom", ["hallway", "bathroom"]),
      ("bathroom", ["hallway"]),
      ("dining room", ["living room", "kitchen"])]),
    
    ("compact_studio",
     [("studio", ["kitchen", "bathroom"]),
      ("kitchen", ["studio"]),
      ("bathroom", ["studio"])]),
    
    ("two_bed",
     [("living room", ["dining room", "kitchen", "hallway"]),
      ("kitchen", ["dining room", "living room"]),
      ("bedroom", ["hallway", "bathroom"]),
      ("bedroom", ["hallway"]),
      ("bathroom", ["hallway"]),
      ("dining room", ["living room", "kitchen"]),
      ("hallway", ["living room", "bedroom", "bathroom"])]),
]

# Area hints (min, max) in square meters
area_hints = {
    "living room": (18, 35),
    "dining room": (10, 20),
    "kitchen": (10, 16),
    "bedroom": (10, 16),
    "bathroom": (4, 8),
    "hallway": (4, 10),
    "studio": (20, 35),
    "storage": (2, 6),
    "pantry": (2, 5),
    "office": (8, 14),
    "balcony": (3, 8),
    "entrance": (3, 8),
}

def area_hint_for(room_type):
    """Get suggested area for a room type."""
    rt = room_type.strip().lower()
    lo, hi = area_hints.get(rt, (8, 18))
    return round((lo + hi) / 2.0, 1)

print(f"Loaded {len(known_graphs)} graph patterns")
print(f"Loaded {len(area_hints)} area hints")

In [ ]:
# Create text descriptions for patterns
def pattern_to_text(name, pairs):
    lines = [f"Pattern: {name}."]
    for room_type, neighbors in pairs:
        if neighbors:
            lines.append(f"Room '{room_type}' is commonly connected to {', '.join([repr(n) for n in neighbors])}.")
        else:
            lines.append(f"Room '{room_type}' may be isolated.")
    return " ".join(lines)

# Create knowledge base prompts
kb_prompts = []
for name, pairs in known_graphs:
    kb_prompts.append(pattern_to_text(name, pairs))

# Add rule snippets
kb_prompts.extend([
    "Kitchens are typically adjacent to dining rooms and often near living rooms.",
    "Bedrooms are usually off a hallway and near bathrooms; bathrooms connect to hallway rather than living spaces.",
    "In compact layouts, consider adding a hallway to mediate connections among bedrooms and bathrooms.",
    "Dining rooms commonly bridge kitchens and living rooms.",
    "Every house should have an entrance that connects to the main living area or hallway.",
])

# Compute embeddings if available
if USE_EMBEDDINGS and model:
    print("Computing knowledge base embeddings...")
    kb_embeddings = model.encode(kb_prompts, convert_to_numpy=True, normalize_embeddings=True)
    print(f"Computed embeddings for {len(kb_prompts)} snippets")
else:
    kb_embeddings = None
    print("Embeddings not available - using text matching fallback")

## Step 4: Graph Utilities

Functions for working with the topologic_fast graph.

In [ ]:
# Room metadata storage (since Dictionary integration is limited)
room_data = {}  # vertex_id -> {type, area, ...}

def vertex_id(vertex):
    """Create a unique ID for a vertex based on coordinates."""
    coords = vertex.Coordinates()
    return f"{coords[0]:.2f}_{coords[1]:.2f}_{coords[2]:.2f}"

def get_room_type(vertex):
    """Get the room type for a vertex."""
    vid = vertex_id(vertex)
    return room_data.get(vid, {}).get('type')

def new_room_vertex(room_type, area=None):
    """Create a new vertex for a room."""
    x = random.uniform(0, 100)
    y = random.uniform(0, 100)
    z = random.uniform(0, 100)
    v = tf.Vertex.ByCoordinates(x, y, z)
    
    # Store metadata
    vid = vertex_id(v)
    room_data[vid] = {
        'type': room_type,
        'area': area if area else area_hint_for(room_type)
    }
    
    return v

def get_vertex_by_type(graph, room_type):
    """Find first vertex matching a room type."""
    room_type = room_type.strip().lower()
    for v in graph.Vertices():
        t = get_room_type(v)
        if t and t.strip().lower() == room_type:
            return v
    return None

def graph_room_types(graph):
    """Get list of room types in the graph."""
    types = []
    for v in graph.Vertices():
        t = get_room_type(v)
        if t:
            types.append(t.strip().lower())
    return sorted(set(types))

def graph_to_text(graph):
    """Convert graph to text description."""
    types = graph_room_types(graph)
    if not types:
        return "Current rooms: none."
    
    parts = [f"Current rooms: {', '.join(types)}."]
    
    # Add adjacency info
    for v in graph.Vertices():
        t = get_room_type(v)
        if not t:
            continue
        adj = graph.AdjacentVertices(v)
        adj_types = [get_room_type(av) for av in adj if get_room_type(av)]
        if adj_types:
            parts.append(f"'{t}' connected to: {', '.join(adj_types)}.")
    
    return " ".join(parts)

print("Graph utilities loaded.")

## Step 5: Retrieval and LLM Integration

In [ ]:
def retrieve_kb_context(graph, k=4):
    """Retrieve relevant KB snippets for the current graph."""
    if not USE_EMBEDDINGS or kb_embeddings is None:
        # Fallback: return first k snippets
        return kb_prompts[:k]
    
    query = graph_to_text(graph)
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims = cosine_similarity(q_emb, kb_embeddings)[0]
    idx = np.argsort(-sims)[:k]
    return [kb_prompts[i] for i in idx]

def examples_from_kb(graph):
    """Generate few-shot examples based on graph state."""
    types = graph_room_types(graph)
    examples = []
    
    # Suggest kitchen if missing and we have dining/living
    if "kitchen" not in types and ("dining room" in types or "living room" in types):
        area = area_hint_for("kitchen")
        examples.append(f"Add 'kitchen' connected to 'dining room', 'living room' area={area}")
    
    # Suggest bathroom if we have bedrooms
    if "bedroom" in types and "bathroom" not in types:
        area = area_hint_for("bathroom")
        examples.append(f"Add 'bathroom' connected to 'hallway' area={area}")
    
    # Suggest hallway if we have bedrooms but no hallway
    if "bedroom" in types and "hallway" not in types:
        area = area_hint_for("hallway")
        examples.append(f"Add 'hallway' connected to 'living room' area={area}")
    
    # Default example
    if not examples:
        examples = ["Add 'storage' with no connections area=4"]
    
    return examples

In [ ]:
def query_llm(prompt, examples):
    """Call Claude API to get graph expansion suggestions."""
    if not USE_LLM or client is None:
        # Mock response for testing
        return "Add 'hallway' connected to 'entrance' area=6\nSTOP"
    
    system_msg = """You are a graph expansion assistant for residential floorplans.

Output rules:
1) Each line is ONE instruction. Allowed forms:
   - Add 'room_type' connected to 'type1', 'type2', ... area=<number>
   - Add 'room_type' with no connections area=<number>
   - Connect 'type_a' to 'type_b'
   - Remove 'room_type'
   - STOP
2) For every Add instruction, you MUST append 'area=<number>' (square meters).
3) Do NOT explain. Output ONLY instruction lines.
"""
    
    user_content = "Examples:\n" + "\n".join(examples) + "\n\n" + prompt
    combined_prompt = f"{system_msg}\n\n{user_content}"
    
    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=1000,
            messages=[{"role": "user", "content": combined_prompt}]
        )
        return response.content[0].text.strip()
    except Exception as e:
        print(f"LLM error: {e}")
        return "STOP"

print("LLM integration loaded.")

## Step 6: Apply Suggestions to Graph

In [ ]:
def parse_area(text):
    """Extract area value from text."""
    m = re.search(r"\barea\s*=\s*([0-9]+(?:\.[0-9]+)?)\b", text, flags=re.IGNORECASE)
    if m:
        return float(m.group(1))
    return None

def sanitize_list_str(list_str):
    """Parse "'a', 'b', 'c'" into ['a', 'b', 'c']."""
    return [s.strip() for s in re.findall(r"'([^']+)'", list_str) if s.strip()]

def apply_suggestion(graph, suggestion):
    """
    Apply LLM suggestions to the graph.
    Returns (new_graph, stop_flag).
    """
    suggestion = (suggestion or "").strip()
    if not suggestion:
        print("  Empty suggestion.")
        return graph, True
    
    lines = [ln.strip() for ln in suggestion.splitlines() if ln.strip()]
    stop_signaled = False
    
    for line in lines:
        l = line.strip()
        
        # STOP
        if l.lower() == "stop":
            print("  STOP signal detected.")
            stop_signaled = True
            continue
        
        # Add 'X' connected to 'a', 'b', ... area=NN
        m_add_conn = re.match(
            r"^add\s+'([^']+)'\s+connected\s+to\s+(.+?)\s*(area\s*=\s*[0-9]+(?:\.[0-9]+)?)?\s*$",
            l, flags=re.IGNORECASE
        )
        if m_add_conn:
            room_type = m_add_conn.group(1).strip()
            neighbor_str = m_add_conn.group(2)
            area_val = parse_area(l)
            
            v_new = new_room_vertex(room_type, area_val)
            graph = graph.AddVertex(v_new)
            print(f"  Added node '{room_type}' with area={area_val} m^2")
            
            neighbors = sanitize_list_str(neighbor_str)
            for nb in neighbors:
                v_target = get_vertex_by_type(graph, nb)
                if v_target:
                    edge = tf.Edge.ByStartVertexEndVertex(v_new, v_target)
                    graph = graph.AddEdge(edge)
                    print(f"    Connected '{room_type}' to '{nb}'")
                else:
                    print(f"    Warning: No node of type '{nb}' found.")
            continue
        
        # Add 'X' with no connections area=NN
        m_add_noc = re.match(
            r"^add\s+'([^']+)'\s+with\s+no\s+connections\s*(area\s*=\s*[0-9]+(?:\.[0-9]+)?)?\s*$",
            l, flags=re.IGNORECASE
        )
        if m_add_noc:
            room_type = m_add_noc.group(1).strip()
            area_val = parse_area(l)
            
            v_new = new_room_vertex(room_type, area_val)
            graph = graph.AddVertex(v_new)
            print(f"  Added node '{room_type}' with area={area_val} m^2 (no connections)")
            continue
        
        # Connect 'a' to 'b'
        m_conn = re.match(r"^connect\s+'([^']+)'\s+to\s+'([^']+)'\s*$", l, flags=re.IGNORECASE)
        if m_conn:
            a = m_conn.group(1).strip()
            b = m_conn.group(2).strip()
            va = get_vertex_by_type(graph, a)
            vb = get_vertex_by_type(graph, b)
            if va and vb:
                edge = tf.Edge.ByStartVertexEndVertex(va, vb)
                graph = graph.AddEdge(edge)
                print(f"  Connected '{a}' to '{b}'")
            else:
                if not va:
                    print(f"  Warning: No node of type '{a}' found.")
                if not vb:
                    print(f"  Warning: No node of type '{b}' found.")
            continue
        
        # Remove 'X'
        m_rem = re.match(r"^remove\s+'([^']+)'\s*$", l, flags=re.IGNORECASE)
        if m_rem:
            rtype = m_rem.group(1).strip().lower()
            v = get_vertex_by_type(graph, rtype)
            if v:
                graph = graph.RemoveVertex(v)
                print(f"  Removed node '{rtype}'")
            else:
                print(f"  Warning: No node of type '{rtype}' to remove.")
            continue
        
        print(f"  Unrecognized instruction: {l}")
    
    return graph, stop_signaled

print("Apply suggestion function loaded.")

## Step 7: User Input

In [ ]:
# Get user requirements
print("How many bedrooms and bathrooms should the house have?")
print("(Example: 'three bedrooms and two bathrooms')")

user_prompt = input("Requirements: ").strip()

if not user_prompt:
    user_prompt = "three bedrooms and two bathrooms"
    print(f"Using default: {user_prompt}")

print(f"\nBuilding floor plan with: {user_prompt}")

## Step 8: Run Graph Expansion

In [ ]:
def initialize_graph(seed_types):
    """Create initial graph with seed rooms."""
    vertices = [new_room_vertex(rt) for rt in seed_types]
    return tf.Graph.ByVerticesEdges(vertices, [])

# Initialize with entrance
graph = initialize_graph(["entrance"])
print(f"Initial graph: {graph_to_text(graph)}")

In [ ]:
def expand_graph_iterative(graph, user_prompt, max_steps=40):
    """
    Iteratively expand the graph using LLM suggestions.
    """
    intermediate_graphs = []
    stop = False
    step = 0
    
    print(f"Maximum steps: {max_steps}")
    
    while not stop and step < max_steps:
        # Build prompt
        current_state = graph_to_text(graph)
        print(f"\nStep {step + 1}: {current_state}")
        
        prompt = (
            "STATELESS: On each turn, IGNORE previous replies and decide ONLY from the current state.\n"
            f"Current state: {current_state}\n\n"
            "Expand the graph of this house by suggesting one additional node. "
            "If you think the graph does not need additional nodes, return STOP. "
            f"The graph should have an entrance, a kitchen, a dining room, a living room, and EXACTLY {user_prompt}."
        )
        
        # Get examples
        examples = examples_from_kb(graph)
        
        # Query LLM
        suggestion = query_llm(prompt, examples)
        print(f"  LLM suggestion: {suggestion[:100]}..." if len(suggestion) > 100 else f"  LLM suggestion: {suggestion}")
        
        # Apply suggestion
        graph, stop = apply_suggestion(graph, suggestion)
        intermediate_graphs.append(graph)
        step += 1
        
        if step == max_steps:
            print("\nMaximum steps reached.")
    
    return graph, intermediate_graphs

# Run expansion
print("\nStarting graph expansion...\n")
final_graph, intermediate_graphs = expand_graph_iterative(graph, user_prompt, max_steps=20)

print(f"\n" + "="*60)
print(f"Final graph: {graph_to_text(final_graph)}")
print(f"Total rooms: {len(final_graph.Vertices())}")
print(f"Total connections: {len(final_graph.Edges())}")

## Step 9: Visualize the Result

In [ ]:
# Color map for room types
room_type_to_color = {
    "bedroom": "lightblue",
    "bathroom": "lightgray",
    "kitchen": "peachpuff",
    "dining room": "khaki",
    "living room": "mediumseagreen",
    "hallway": "darkslategray",
    "entrance": "gold",
    "storage": "saddlebrown",
    "office": "lightsteelblue",
    "pantry": "linen",
}

def visualize_graph(graph, title="Floor Plan Graph"):
    """Visualize the graph using Plotly."""
    fig = go.Figure()
    
    vertices = graph.Vertices()
    
    # Assign positions in a grid layout
    n = len(vertices)
    cols = max(3, int(math.ceil(math.sqrt(n))))
    positions = {}
    
    for i, v in enumerate(vertices):
        row = i // cols
        col = i % cols
        positions[vertex_id(v)] = (col * 3, row * 3)
    
    # Draw edges
    edges = graph.Edges()
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = positions.get(vertex_id(edge_verts[0]), (0, 0))
            p2 = positions.get(vertex_id(edge_verts[1]), (0, 0))
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='red', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    for v in vertices:
        vid = vertex_id(v)
        pos = positions.get(vid, (0, 0))
        rtype = get_room_type(v) or "unknown"
        color = room_type_to_color.get(rtype.lower(), "black")
        area = room_data.get(vid, {}).get('area', 0)
        
        fig.add_trace(go.Scatter(
            x=[pos[0]],
            y=[pos[1]],
            mode='markers+text',
            marker=dict(size=30, color=color, line=dict(color='black', width=2)),
            text=[rtype],
            textposition='top center',
            textfont=dict(size=10),
            name=f"{rtype} ({area}m^2)",
            hovertext=f"{rtype}\nArea: {area} m^2",
            hoverinfo='text'
        ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        width=800,
        height=600,
        showlegend=True
    )
    
    return fig

fig = visualize_graph(final_graph, f"Generated Floor Plan - {user_prompt}")
fig.show()

## Step 10: Adjacency Matrix

In [ ]:
# Get adjacency matrix
adj_matrix = final_graph.AdjacencyMatrix()

# Get room labels
room_labels = [get_room_type(v) or f"Room {i}" for i, v in enumerate(final_graph.Vertices())]

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=adj_matrix,
    x=room_labels,
    y=room_labels,
    colorscale='RdYlGn_r',
    text=adj_matrix,
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorbar=dict(title='Connected')
))

fig.update_layout(
    title='Room Adjacency Matrix',
    xaxis=dict(title='', tickangle=45, tickfont=dict(size=9)),
    yaxis=dict(title='', autorange='reversed', tickfont=dict(size=9)),
    width=700,
    height=600
)

fig.show()

## Step 11: Room Summary

In [ ]:
print("\nGenerated Floor Plan Summary")
print("=" * 60)

total_area = 0
for i, v in enumerate(final_graph.Vertices()):
    vid = vertex_id(v)
    data = room_data.get(vid, {})
    rtype = data.get('type', 'unknown')
    area = data.get('area', 0)
    degree = final_graph.VertexDegree(v)
    total_area += area
    
    print(f"  {i+1:2d}. {rtype:20s} | Area: {area:6.1f} m^2 | Connections: {degree}")

print("=" * 60)
print(f"Total floor area: {total_area:.1f} m^2 ({total_area * 10.764:.0f} sq ft)")
print(f"Number of rooms: {len(final_graph.Vertices())}")
print(f"Number of connections: {len(final_graph.Edges())}")

## Summary

This notebook demonstrated GraphRAG-style floor plan generation:

1. **Knowledge Base**: Room adjacency patterns and area hints
2. **Embedding Retrieval**: Finding relevant patterns using sentence-transformers
3. **LLM Integration**: Using Claude to suggest graph expansions
4. **Graph Building**: Using topologic_fast to construct the room graph
5. **Visualization**: Plotly-based graph and adjacency matrix views

### Not Yet Implemented in topologic_fast

The following topologicpy features are not yet available:

- `Dictionary` integration with vertices (used `room_data` dict instead)
- `Graph.Reshape()` - Force-directed layout
- `Graph.Intersect()` - Graph intersection
- `Graph.DegreeCentrality()` - Centrality metrics
- `Graph.RemoveIsolatedVertices()` - Cleanup isolated nodes
- `Plotly.DataByGraph()` - Native graph visualization
- `Plotly.ExportToImage()` - Save visualization as image

### Requirements

- `topologic_fast`
- `anthropic` (for Claude API)
- `sentence-transformers` (for embeddings)
- `scikit-learn` (for cosine similarity)
- `plotly` (for visualization)